# 11 — Calibration for LLMs

Temperature scaling, RLHF effects, and post-hoc calibration improve LLM reliability.

## Temperature Scaling for LLMs and RLHF Calibration Effects

**Temperature scaling** (Guo et al., 2017) is a simple post-hoc calibration method:
$$\hat{p}(y|x) = \text{softmax}(z/T)$$
A single temperature parameter *T* is fit on a calibration set to minimise NLL. *T>1* softens predictions (reduces overconfidence); *T<1* sharpens them.

**RLHF calibration effects** are well-documented:
- Base LLMs trained on next-token prediction have reasonably calibrated probabilities
- After RLHF (preference fine-tuning), models become more verbose and confident-sounding
- This verbal overconfidence often reflects training reward signals that favoured authoritative-sounding answers
- Calibration of verbalized confidence often *worsens* after RLHF
- However, calibration of token probabilities may improve if the reward model is itself calibrated

**Calibration after fine-tuning**: when fine-tuning for downstream tasks, re-calibrate the model's confidence on the task-specific calibration set. Temperature T found for base model may not transfer.

**Reliable AI** requires jointly optimising for accuracy AND calibration — the 'reliable AI tax' is the cost of not deploying overconfident models.

In [ ]:
# LLM calibration pipeline
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)

# Simulate pre-RLHF and post-RLHF models with different calibration
X, y = make_classification(n_samples=2000, n_features=10, n_classes=4,
                            n_informative=6, n_redundant=2, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y)

class Classifier(nn.Module):
    def __init__(self, sharpening=1.0):
        super().__init__()
        self.sharpening = sharpening
        self.net = nn.Sequential(
            nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 4)
        )
    def forward(self, x):
        return self.net(x) * self.sharpening

# Train base model (well-calibrated)
base_model = Classifier(sharpening=1.0)
opt = torch.optim.Adam(base_model.parameters(), lr=1e-3)
for _ in range(300):
    loss = nn.CrossEntropyLoss()(base_model(X[:1600]), y[:1600])
    opt.zero_grad(); loss.backward(); opt.step()

# Simulate RLHF by sharpening logits (overconfident post-RLHF)
rlhf_model = Classifier(sharpening=2.5)
rlhf_model.net.load_state_dict(base_model.net.state_dict())

# Calibration set
X_cal, y_cal = X[1600:1800], y[1600:1800]
X_test, y_test = X[1800:], y[1800:]

def compute_ece(model, X_data, y_data, n_bins=10):
    with torch.no_grad():
        probs = model(X_data).softmax(dim=-1).numpy()
        labels = y_data.numpy()
    confs = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    corrects = (preds == labels).astype(float)

    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confs >= lo) & (confs < hi)
        if mask.sum() > 0:
            ece += mask.mean() * abs(corrects[mask].mean() - confs[mask].mean())
    return ece

print('Calibration comparison:')
print(f'  Base model ECE: {compute_ece(base_model, X_test, y_test):.4f}')
print(f'  RLHF model ECE: {compute_ece(rlhf_model, X_test, y_test):.4f} (overconfident)')

# Temperature scaling on RLHF model
def find_temperature(model, X_cal, y_cal):
    with torch.no_grad():
        logits = model.net(X_cal)
    best_T, best_nll = 1.0, float('inf')
    for T in np.arange(0.1, 5.0, 0.1):
        nll = nn.CrossEntropyLoss()(logits / T, y_cal).item()
        if nll < best_nll:
            best_nll = nll
            best_T = T
    return best_T

T_opt = find_temperature(rlhf_model, X_cal, y_cal)

# Apply temperature-scaled RLHF model
class TempScaledModel(nn.Module):
    def __init__(self, model, T):
        super().__init__()
        self.model = model
        self.T = T
    def forward(self, x):
        return self.model(x) / self.T

scaled_rlhf = TempScaledModel(rlhf_model, T_opt)
print(f'  RLHF + temp scaling (T={T_opt:.1f}) ECE: {compute_ece(scaled_rlhf, X_test, y_test):.4f}')